In [2]:
import pandas as pd

# Carrega o ficheiro completo fipe_cars.csv
df = pd.read_csv('fipe_cars.csv')

# Mostra as colunas disponíveis para confirmar
print("Colunas disponíveis no dataset:", df.columns.tolist())

# Exporta os dados para um novo ficheiro limpo
df.to_csv('tabela_fipe_limpa.csv', index=False, encoding='utf-8')
print("Ficheiro 'tabela_fipe_limpa.csv' gerado com sucesso!")

Colunas disponíveis no dataset: ['year_of_reference', 'month_of_reference', 'fipe_code', 'authentication', 'brand', 'model', 'fuel', 'gear', 'engine_size', 'year_model', 'avg_price_brl']
Ficheiro 'tabela_fipe_limpa.csv' gerado com sucesso!


In [3]:
# Mostra as primeiras 5 linhas do dataset para conhecermos os dados
df.head()

,year_of_reference,month_of_reference,fipe_code,authentication,brand,model,fuel,gear,engine_size,year_model,avg_price_brl
0,2021,January,038001-6,tlp4qry07m,Acura,NSX 3.0,Gasoline,manual,3.0,1995,40374.0
1,2021,January,038001-6,s1wksdv9by,Acura,NSX 3.0,Gasoline,manual,3.0,1994,38939.0
2,2021,January,038001-6,skrbcfnkch,Acura,NSX 3.0,Gasoline,manual,3.0,1993,37648.0
3,2021,January,038001-6,rxzh76d5db,Acura,NSX 3.0,Gasoline,manual,3.0,1992,35962.0
4,2021,January,038001-6,qrm322tpd8,Acura,NSX 3.0,Gasoline,manual,3.0,1991,32863.0


In [4]:
# 1. Selecionar as colunas mais relevantes para prever o preço
colunas_modelo = ['brand', 'gear', 'fuel', 'engine_size', 'year_model', 'avg_price_brl']
df_modelo = df[colunas_modelo].dropna()

# 2. Converter variáveis categóricas (texto) em variáveis numéricas (One-Hot Encoding)
df_processado = pd.get_dummies(df_modelo, columns=['brand', 'gear', 'fuel'], drop_first=True)

# 3. Separar as variáveis explicativas (X) da variável que queremos prever (y)
X = df_processado.drop('avg_price_brl', axis=1)
y = df_processado['avg_price_brl']

print("Tamanho do conjunto X (recursos):", X.shape)
print("Tamanho do conjunto y (preços):", y.shape)
print("\nDados preparados com sucesso!")

Tamanho do conjunto X (recursos): (599007, 91)
Tamanho do conjunto y (preços): (599007,)

Dados preparados com sucesso!


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Dividir os dados em Conjunto de Treino (80%) e Teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Dados divididos para treino e teste com sucesso!")

# 2. Criar e treinar o modelo Random Forest (com n_estimators menor para rodar mais rápido)
print("A treinar o modelo Random Forest (isto pode demorar alguns segundos)...")
modelo = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)
print("Modelo treinado com sucesso!")

# 3. Fazer previsões no conjunto de teste e avaliar o resultado
y_pred = modelo.predict(X_test)

# Calcular métricas de precisão
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n--- Resultados do Modelo ---")
print(f"Erro Absoluto Médio (MAE): R$ {mae:.2f}")
print(f"Precisão do Modelo (R² Score): {r2 * 100:.2f}%")

Dados divididos para treino e teste com sucesso!
A treinar o modelo Random Forest (isto pode demorar alguns segundos)...
Modelo treinado com sucesso!

--- Resultados do Modelo ---
Erro Absoluto Médio (MAE): R$ 16693.82
Precisão do Modelo (R² Score): 95.77%


In [5]:
import pandas as pd

# 1. Defina os dados do carro que quer testar:
# Exemplo: Marca 'Fiat', Câmbio 'manual', Combustível 'Gasoline', Motor 1.0, Ano 2020
carro_teste = {
    'brand': 'Fiat',
    'gear': 'manual',        # 'manual' ou 'automatic'
    'fuel': 'Gasoline',      # 'Gasoline', 'Diesel', etc.
    'engine_size': 1.0,      # ex: 1.0, 1.6, 2.0
    'year_model': 2020       # ano do modelo
}

# 2. Converte os dados para o mesmo formato usado no treino
df_carro = pd.DataFrame([carro_teste])
df_carro_proc = pd.get_dummies(df_carro, columns=['brand', 'gear', 'fuel'])

# Alinha as colunas com as do treino (preenche colunas em falta com 0)
df_carro_proc = df_carro_proc.reindex(columns=X.columns, fill_value=0)

# 3. Faz a previsão do preço
preco_previsto = modelo.predict(df_carro_proc)[0]

print(f"--- Previsão do Modelo ---")
print(f"Carro: {carro_teste['brand']} {carro_teste['engine_size']} ({carro_teste['year_model']})")
print(f"Preço Estimado pela FIPE: R$ {preco_previsto:,.2f}")

--- Previsão do Modelo ---
Carro: Fiat 1.0 (2020)
Preço Estimado pela FIPE: R$ 48,242.86


In [9]:
import pandas as pd

# 1. Defina os dados do veículo que quer consultar
carro_consulta = {
    'brand': 'Fiat',
    'gear': 'manual',        # 'manual' ou 'automatic'
    'fuel': 'Gasoline',      # 'Gasoline', 'Diesel', etc.
    'engine_size': 1.0,      # ex: 1.0, 1.6, 2.0
    'year_model': 2020       # ano do modelo
}

# 2. Processar os dados para o modelo
df_carro = pd.DataFrame([carro_consulta])
df_carro_proc = pd.get_dummies(df_carro, columns=['brand', 'gear', 'fuel'])
df_carro_proc = df_carro_proc.reindex(columns=X.columns, fill_value=0)

# 3. Calcular preço estimado e o menor valor recomendado (ex: 7% de margem mínima)
preco_fipe_estimado = modelo.predict(df_carro_proc)[0]
margem_desconto = 0.20  # 7% de margem para o menor preço aceitável
menor_valor_possivel = preco_fipe_estimado * (1 - margem_desconto)

print(f"--- Avaliação de Preço ---")
print(f"Veículo: {carro_consulta['brand']} {carro_consulta['engine_size']} ({carro_consulta['year_model']})")
print(f"Preço Médio FIPE: R$ {preco_fipe_estimado:,.2f}")
print(f"Menor valor recomendado a pedir: R$ {menor_valor_possivel:,.2f}")

--- Avaliação de Preço ---
Veículo: Fiat 1.0 (2020)
Preço Médio FIPE: R$ 48,242.86
Menor valor recomendado a pedir: R$ 38,594.29


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# 1. Filtrar o dataset original apenas para a marca 'Fiat'
# (Nota: certifique-se de que o nome da marca no dataset está como 'Fiat')
df_fiat = df[df['brand'] == 'Fiat'].copy()

print(f"Total de carros da Fiat encontrados: {len(df_fiat)}")

# 2. Selecionar as colunas relevantes, incluindo o 'model'
colunas_modelo = ['model', 'gear', 'fuel', 'engine_size', 'year_model', 'avg_price_brl']
df_fiat_modelo = df_fiat[colunas_modelo].dropna()

# 3. Converter texto em números (One-Hot Encoding) para os modelos da Fiat
df_fiat_proc = pd.get_dummies(df_fiat_modelo, columns=['model', 'gear', 'fuel'], drop_first=True)

# 4. Separar variáveis independentes (X) e o preço alvo (y)
X_fiat = df_fiat_proc.drop('avg_price_brl', axis=1)
y_fiat = df_fiat_proc['avg_price_brl']

# 5. Dividir em treino (80%) e teste (20%)
X_train_fiat, X_test_fiat, y_train_fiat, y_test_fiat = train_test_split(X_fiat, y_fiat, test_size=0.2, random_state=42)

# 6. Treinar o Random Forest focado apenas na Fiat
print("A treinar o modelo para os veículos Fiat...")
modelo_fiat = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
modelo_fiat.fit(X_train_fiat, y_train_fiat)
print("Modelo da Fiat treinado com sucesso!")

Total de carros da Fiat encontrados: 55087
A treinar o modelo para os veículos Fiat...
Modelo da Fiat treinado com sucesso!


In [8]:
# Obter a lista única de todos os modelos da Fiat ordenados alfabeticamente
modelos_fiat = sorted(df[df['brand'] == 'Fiat']['model'].unique())

print(f"Total de modelos diferentes da Fiat: {len(modelos_fiat)}\n")
print("Lista de modelos disponíveis:")
for modelo in modelos_fiat:
    print(f"- {modelo}")

Total de modelos diferentes da Fiat: 531

Lista de modelos disponíveis:
- 500 ABARTH MULTIAIR 1.4 TB 16V 3p
- 500 Cabrio Dualogic Flex 1.4 8V
- 500 Cabrio Flex 1.4 8V Mec.
- 500 Cabrio/500 Coupe Gucci/Flex 1.4 Aut.
- 500 Cult 1.4 Flex 8V EVO Dualogic
- 500 Cult 1.4 Flex 8V EVO Mec.
- 500 LOUNGE 1.4 16V 100cv Dualogic
- 500 LOUNGE 1.4 16V 100cv Mec.
- 500 Lounge Air 1.4/ 1.4 Flex 16V Aut.
- 500 SPORT 1.4 16V 100cv Dualogic
- 500 SPORT 1.4 16V 100cv Mec.
- 500 Sport Air 1.4 16V/1.4 Flex 16V Aut.
- 500 Sport Air 1.4 16V/1.4 Flex Mec.
- ARGO 1.0 6V Flex.
- ARGO DRIVE 1.0 6V Flex
- ARGO DRIVE 1.3 8V Flex
- ARGO DRIVE GSR 1.3 8V Flex
- ARGO DRIVE S-DESIGN 1.3 8V Flex
- ARGO HGT 1.8 16V Flex Aut.
- ARGO HGT 1.8 16V Flex Mec.
- ARGO PRECISION 1.8 16V Flex Aut.
- ARGO PRECISION 1.8 16V Flex Mec.
- ARGO TREKKING 1.3 8V Flex
- ARGO TREKKING 1.8 16V Flex Aut.
- Brava ELX 1.6 16V 4p
- Brava HGT 1.8 mpi 16V 4p
- Brava SX 1.6 16V 4p
- Bravo ABSOLUTE 1.8 Flex 16V 5p
- Bravo ABSOLUTE Dualogic 1.8 Flex 

In [9]:
import pandas as pd

# 1. Defina os dados do carro da Fiat que quer testar
# (Certifique-se de que o nome do modelo está exatamente como escrito no dataset, ex: 'Uno', 'Argo', 'Pulse', etc.)
carro_teste_fiat = {
    'model': 'Uno',          # Nome do modelo específico
    'gear': 'manual',        # 'manual' ou 'automatic'
    'fuel': 'gasoline',      # 'Gasoline', 'Ethanol', etc.
    'engine_size': 1.0,      # ex: 1.0, 1.4
    'year_model': 2002       # Ano do modelo
}

# 2. Processar os dados para o formato do modelo da Fiat
df_teste = pd.DataFrame([carro_teste_fiat])
df_teste_proc = pd.get_dummies(df_teste, columns=['model', 'gear', 'fuel'])

# Alinha as colunas com as usadas no treino da Fiat
df_teste_proc = df_teste_proc.reindex(columns=X_fiat.columns, fill_value=0)

# 3. Fazer a previsão do preço FIPE e calcular o menor valor (15% abaixo)
preco_fipe = modelo_fiat.predict(df_teste_proc)[0]
margem_desconto = 0.15
menor_valor = preco_fipe * (1 - margem_desconto)

# 4. Mostrar o resultado
print(f"--- Avaliação de Preço: Fiat ---")
print(f"Modelo: Fiat {carro_teste_fiat['model']} {carro_teste_fiat['engine_size']} ({carro_teste_fiat['year_model']})")
print(f"Preço Médio FIPE Estimado: R$ {preco_fipe:,.2f}")
print(f"Menor valor recomendado a pedir (15% abaixo): R$ {menor_valor:,.2f}")

--- Avaliação de Preço: Fiat ---
Modelo: Fiat Uno 1.0 (2002)
Preço Médio FIPE Estimado: R$ 9,622.67
Menor valor recomendado a pedir (15% abaixo): R$ 8,179.27


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# 1. Garantir que temos listas limpas e únicas dos dados da marca
marcas_disponive = sorted(df['brand'].unique().tolist())

# 2. Criar os widgets interativos
dropdown_marca = widgets.Dropdown(
    options=marcas_disponive,
    value='Fiat' if 'Fiat' in marcas_disponive else marcas_disponive[0],
    description='Marca:',
    style={'description_width': 'initial'}
)

dropdown_modelo = widgets.Dropdown(
    options=[], 
    description='Modelo:',
    style={'description_width': 'initial'}
)

dropdown_ano = widgets.Dropdown(
    options=[], 
    description='Ano:',
    style={'description_width': 'initial'}
)

dropdown_motor = widgets.Dropdown(
    options=[], 
    description='Motor:',
    style={'description_width': 'initial'}
)

dropdown_cambio = widgets.Dropdown(
    options=['manual', 'automatic'], 
    description='Câmbio:',
    style={'description_width': 'initial'}
)

dropdown_combustivel = widgets.Dropdown(
    options=['Gasoline', 'Flex', 'Ethanol', 'Diesel'], 
    description='Combustível:',
    style={'description_width': 'initial'}
)

botao_calcular = widgets.Button(
    description='Calcular FIPE & Mínimo',
    button_style='success',
    icon='calculator'
)

output_resultado = widgets.Output()

# --- Funções para atualizar os menus em cascata automaticamente ---

def atualizar_modelos(change):
    marca_selecionada = dropdown_marca.value
    modelos = sorted(df[df['brand'] == marca_selecionada]['model'].unique().tolist())
    dropdown_modelo.options = modelos
    if modelos:
        dropdown_modelo.value = modelos[0]

def atualizar_anos_e_motores(change):
    marca = dropdown_marca.value
    modelo = dropdown_modelo.value
    
    sub_df = df[(df['brand'] == marca) & (df['model'] == modelo)]
    
    anos = sorted(sub_df['year_model'].unique().tolist(), reverse=True)
    dropdown_ano.options = anos
    if anos:
        dropdown_ano.value = anos[0]
        
    motores = sorted(sub_df['engine_size'].unique().tolist())
    dropdown_motor.options = motores
    if motores:
        dropdown_motor.value = motores[0]

# Conectar os eventos de mudança
dropdown_marca.observe(atualizar_modelos, names='value')
dropdown_modelo.observe(atualizar_anos_e_motores, names='value')

# Disparar a primeira atualização para preencher os campos iniciais
if marcas_disponive:
    atualizar_modelos(None)

# --- Função executada ao clicar no botão ---
def on_botao_clicked(b):
    with output_resultado:
        clear_output()
        
        marca = dropdown_marca.value
        modelo = dropdown_modelo.value
        ano = dropdown_ano.value
        motor = dropdown_motor.value
        cambio = dropdown_cambio.value
        combustivel = dropdown_combustivel.value
        
        # Buscar o preço real correspondente no dataset
        match = df[
            (df['brand'] == marca) & 
            (df['model'] == modelo) & 
            (df['year_model'] == ano) &
            (df['engine_size'] == motor)
        ]
        
        if len(match) > 0:
            preco_fipe = match['avg_price_brl'].mean()
        else:
            match_fallback = df[(df['brand'] == marca) & (df['model'] == modelo) & (df['year_model'] == ano)]
            if len(match_fallback) > 0:
                preco_fipe = match_fallback['avg_price_brl'].mean()
            else:
                preco_fipe = df[(df['brand'] == marca) & (df['model'] == modelo)]['avg_price_brl'].mean()
        
        margem_desconto = 0.15
        menor_valor = preco_fipe * (1 - margem_desconto)
        
        print("========================================")
        print(" AVALIAÇÃO FIPE INTERATIVA")
        print("========================================")
        print(f" Veículo: {marca} {modelo} ({ano})")
        print(f" Motor: {motor} | Câmbio: {cambio} | Combustível: {combustivel}")
        print(f"----------------------------------------")
        print(f" Preço Médio FIPE: R$ {preco_fipe:,.2f}")
        print(f" Menor valor recomendado (15% abaixo): R$ {menor_valor:,.2f}")
        print("========================================")

botao_calcular.on_click(on_botao_clicked)

# Exibir a interface no Jupyter Notebook
display(widgets.VBox([
    widgets.HTML("<h3>Selecione os dados do veículo:</h3>"),
    dropdown_marca,
    dropdown_modelo,
    dropdown_ano,
    dropdown_motor,
    dropdown_cambio,
    dropdown_combustivel,
    widgets.HTML("<br>"),
    botao_calcular,
    output_resultado
]))